In [ ]:
import polars as pl
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Iterable, Optional, Callable, List, Any

In [ ]:
# Roll your own  FrozenSet

class Label:
    def __init__(self, **kwargs):
        self._data = dict(kwargs)
        self._hash = hash(frozenset(self._data.items()))

    def __getitem__(self, key):
        return self._data[key]

    def __iter__(self):
        return iter(self._data)

    def items(self):
        return self._data.items()

    def keys(self):
        return self._data.keys()

    def values(self):
        return self._data.values()

    def matches(self, **kwargs):
        for k, v in kwargs.items():
            if k not in self._data:
                raise ValueError(f"Trying to match on unknown namespace {k}")
            elif self._data[k] != v:
                return False
        return True

    def drop(self, *keys):
        new_mapping = {k: v for k, v in self._data.items() if k not in keys}
        return Label(**new_mapping)

    def __eq__(self, other):
        if not isinstance(other, Label):
            return NotImplemented
        return self._data == other._data

    def __hash__(self):
        return self._hash

    def __repr__(self):
        return f"Label({self._data})"

    def column(self):
        return str(self._data)

In [ ]:
class PomapNode(ABC):
    @property
    @abstractmethod
    def children(self) -> Iterable["PomapNode"]:
        """Return iterable of child nodes."""
        ...

In [ ]:
@dataclass
class Leaf(PomapNode):
    label: str
    factory: Callable

    @property
    def children(self):
        return []


In [ ]:
@dataclass
class Lift(PomapNode):
    child: PomapNode
    atomics: Iterable[pl.DataType]
    train_mask_for_label: Callable[pl.DataType, pl.Expr]
    test_mask_for_label: Callable[pl.DataType, pl.Expr]
    name: Optional[str] = None

    def __post_init__(self):
        if self.name is None:
            self.name = f"Lift: {self.atomics}"
        self.atomics = set(self.atomics)

    @property
    def children(self) -> Iterable["PomapNode"]:
        return [self.child]

In [ ]:
@dataclass
class Ensemble(PomapNode):
    models: Iterable[PomapNode]
    name = 'Ensemble'

    @property
    def children(self):
        return self.models

### Define interpreter

In [ ]:
def _print_tree(node: PomapNode, prefix='', is_root=True) -> str:
    # Leaf formatting
    if not hasattr(node, "children") or not node.children:
        label = getattr(node, "label", str(node))
        if is_root:
            return label
        return f"{prefix}└── {label}"

    # Internal node
    if is_root:
        lines = [f"{node.name}"]
    else:
        lines = [f"{prefix}└── {node.name}"]

    for i, child in enumerate(node.children):
        is_last = i == len(node.children) - 1
        next_prefix = prefix + "    "
        lines.append(_print_tree(child, next_prefix, is_root=False))

    return "\n".join(lines)


def _validate_tree(node: PomapNode):
    # TODO Add namespace checking - note there's something a bit funny where
    # 'Ensemble' is used as the name for all Ensembles, but that doesn't actually
    # feed into the labels, so won't cause any clashes. I think we just need to disambiguate
    # The namespace name from the _print_tree name.
    ...
        

def _collect_labels(root: PomapNode) -> List[Label]:
    match root:
        case Leaf(label=l):
            yield Label(leaf=l)
            
        case Lift(child=child, atomics=atomics, name=name):
            # Under a lift, we will take the cartesian product
            # Of the existing labels with the lift atomics
            for child_label in _collect_labels(child):
                for atomic in atomics:
                        yield Label(**{**child_label, name: atomic})

        case Ensemble(models=models):
            for model in models:
                yield from _collect_labels(model)
    
        case _:
            return 


def _collect_leaves(node: PomapNode) -> List[Label]:
    match node.children:
        case []:
            yield node
        case children:
            for child in children:
                yield from _collect_leaves(child)

# def _get_train_mask_for_label(node: PomapNode, label: Label) -> pl.Expr:
#     match node:
#         case leaf if isinstance(leaf, Leaf):
#             return True
            
#         case Lift(child=child, name=name, train_mask_for_label=train_mask_for_label):
#             # Split the label into the part that's relevant for the lift
#             # and the part that is relevant for the rest of the tree.
#             lift_label, child_label = label[name], label.drop(name)
#             return train_mask_for_label(lift_label) & _get_train_mask_for_label(child, child_label)

#         case _:
#             raise NotImplementedError(f"Not implemented for node type {node.name}")

# def _get_test_mask_for_label(node: PomapNode, label: Label) -> pl.Expr:
#     match node:
#         case leaf if isinstance(leaf, Leaf):
#             return True
            
#         case Lift(child=child, name=name, test_mask_for_label=test_mask_for_label):
#             # Split the label into the part that's relevant for the lift
#             # and the part that is relevant for the rest of the tree.
#             lift_label, child_label = label[name], label.drop(name)
#             return test_mask_for_label(lift_label) & _get_test_mask_for_label(child, child_label)

#         case _:
#             raise NotImplementedError(f"Not implemented for node type {node.name}")

def _get_train_df_for_label(node: PomapNode, df: pl.DataFrame, label: Label) -> pl.DataFrame:

    match node:
        case Leaf():
            return df
            
        case Lift(child=child, name=name, train_mask_for_label=train_mask_for_label):
            # In a lift, we apply the mask specified in the lift
            # To the train df returned by the child 

            # First, we plit the label into the part that's relevant for the lift
            # and the part that is relevant for the rest of the tree.
            lift_label, child_label = label[name], label.drop(name)
            mask_expr = train_mask_for_label(lift_label)
            return _get_train_df_for_label(child, df, label=child_label).filter(mask_expr)

        case Ensemble(models):
            # In an ensemble, we just pass through the dataframe
            # from the appropriate child. The correct child is the one
            # That matches the label.
            for child in models:
                if label in _collect_labels(child):
                    return _get_train_df_for_label(child, df, label=label)

        case _:
            raise NotImplementedError(f"Not implemented for node type {node.name}")
                
    return df.filter(train_expr)


# TODO note this can be combined with the method above using one additional arg
def _get_test_df_for_label(node: PomapNode, df: pl.DataFrame, label: Label) -> pl.DataFrame:
        match node:
            case Leaf():
                return df
                
            case Lift(child=child, name=name, test_mask_for_label=test_mask_for_label):
                # In a lift, we apply the mask specified in the lift
                # To the train df returned by the child 
    
                # First, we plit the label into the part that's relevant for the lift
                # and the part that is relevant for the rest of the tree.
                lift_label, child_label = label[name], label.drop(name)
                mask_expr = test_mask_for_label(lift_label)
                return _get_test_df_for_label(child, df, label=child_label).filter(mask_expr)
    
            case Ensemble(models):
                # In an ensemble, we just pass through the dataframe
                # from the appropriate child. The correct child is the one
                # That matches the label.
                for child in models:
                    if label in _collect_labels(child):
                        return _get_test_df_for_label(child, df, label=label)
    
            case _:
                raise NotImplementedError(f"Not implemented for node type {node.name}")
    


def _fit(node: PomapNode, df: pl.DataFrame) -> dict:
    models = {}
    labels = _collect_labels(node)

    # Create a dictionary of leaves, indexed by their atomic labels
    leaves = _collect_leaves(node)
    leaves = {leaf.label: leaf for leaf in leaves}

    for label in labels:
        leaf_label = label['leaf']
        model = leaves[leaf_label].factory()

        train_df = _get_train_df_for_label(node, df, label)
        model.fit(train_df)

        models[label] = model

    return models

def _predict(node: PomapNode, models: dict, df: pl.DataFrame):
    if '__pomap_row_index' in df.columns:
        raise ValueError('Trying to create column __pomap_row_index but it already exists')

    df = df.with_row_index(name='__pomap_row_index')

    labels = _collect_labels(node)
    for label in labels:
        test_df = _get_test_df_for_label(node, df, label)
        predictions = models[label].predict(test_df).rename(label.column())
      
        test_df = test_df.with_columns(predictions)
        
        df = df.join(
            test_df.select('__pomap_row_index', label.column()),
            on='__pomap_row_index',
            coalesce=True,
            how='left'
        )

    return df

### Tests

In [ ]:
from scipy import stats

In [ ]:
@dataclass
class LinearRegression:
    x_column: str
    y_column: str
    intercept: float = None
    beta: float = None
    
    def fit(self, df: pl.DataFrame) -> None:
        x = df[self.x_column]
        y = df[self.y_column]

        lr = stats.linregress(x, y)
        self.intercept = lr[0]
        self.beta = lr[1]

    def predict(self, df: pl.DataFrame) -> pl.Series:
        return  df[self.x_column]*self.beta + self.intercept
        

In [ ]:
class BasicLift(Lift):
     
    def train_mask_for_label(self, label) -> pl.Expr:
        return pl.col('category') == pl.lit(label)

    def test_mask_for_label(self, label) -> pl.Expr:
        return pl.col('category') == pl.lit(label)

In [ ]:
model = Leaf(
    label='lr',
    factory = lambda: LinearRegression(x_column='x', y_column='y')
            )

model2 = Leaf(
    label='lr2',
    factory = lambda:LinearRegression(x_column='y', y_column='y')
)


lifted = Lift(
        model,
        atomics=['a', 'b', 'c'],
        train_mask_for_label=lambda atomic: pl.col('category') == pl.lit(atomic),
        test_mask_for_label= lambda atomic: pl.col('category') == pl.lit(atomic)
        )

ensembled = Ensemble([model, model2])
big_ensembled = Ensemble([model, lifted])

In [ ]:
print(_print_tree(lifted))

In [ ]:
print(_print_tree(ensembled))
print(_print_tree(big_ensembled))

In [ ]:
list(_collect_labels(model))

In [ ]:
list(_collect_labels(lifted))

In [ ]:
list(_collect_labels(ensembled))

In [ ]:
list(_collect_labels(big_ensembled))

In [ ]:
# # # Create some real sample data for testing 
df_a = pl.DataFrame(
    {
    'x': [1., 1.5, 2.],
    'y': [2., 3., 4.],
    'category': ['a', 'a', 'a']
    }
)

df_b = pl.DataFrame(
    {
    'x': [1., 1.5, 2.],
    'y': [-2., -3., -4.],
    'category': ['b', 'b', 'b']
    }
)

df_c = pl.DataFrame(
    {
    'x': [1., 1.5, 2.],
    'y': [4., 6., 8.],
    'category': ['c', 'c', 'c']
    }
)

df = pl.concat([df_a, df_b, df_c])

In [ ]:
one_lr = model.factory()
one_lr.fit(df)

In [ ]:
# Check train df of lift works appropriately
a_label = Label(**{'leaf': 'lr', "Lift: ['a', 'b', 'c']": 'a'})
_get_train_df_for_label(lifted, df, a_label)

In [ ]:
# Also, train_df of ensemble
_get_train_df_for_label(big_ensembled, df, a_label)

In [ ]:
ll = next(_collect_labels(lifted))
ll in _collect_labels(lifted)

In [ ]:
# Test fitting and prediction

lifted_models = _fit(lifted, df)

_predict(lifted, lifted_models, df)

### Define user interface

In [ ]:
class Model:
    # Type used for all Pomap model outputs
    ...

In [ ]:
# Start here - it's annoying but I think we need to force a 'label'
# Onto the model class itself. That way, I don't have to do this extra step of adding
# Labels, which mucks up the ergonomics

def lift(
    model: type, # Should have constructor, fit, predict, label
    train_model_expr:  Callable[pl.DataType, bool],
    test_model_expr:  Callable[pl.DataType, bool],
    atomics: List[pl.DataType],
    lift_name=None,
    ) -> Model:

    if not isinstance(model, PomapNode):
        if not model_label:
             child = Leaf(model)
    else:
       child=model

def ensemble(
    models: List,
    name=None
    ) -> Model:
    ...
    
    